# Responses API 的 MCP 工具使用指南

构建智能体应用通常需要连接外部服务。传统上，这是通过函数调用来实现的：每个操作都要从模型往返到你的后端，再到外部服务，等待响应，最后将结果返回给模型。这个过程引入了多次网络跳转和显著的延迟，使得扩展和管理都很繁琐。

Responses API 中托管的模型上下文协议（Model Context Protocol，MCP）工具让这一切变得更简单。你无需手动将每个函数调用连接到特定服务，只需配置一次模型，让它指向一个 MCP 服务器（也可以是多个！）。该服务器充当集中式的工具宿主，对外暴露“搜索商品目录”或“将商品加入购物车”等标准命令。这使得编排更简单，工具也能集中管理。借助 MCP，模型直接与 MCP 服务器交互，从而降低延迟并省去后端协调。

## MCP 工具简化的用例

MCP 显著降低了构建与外部服务交互的产品的门槛，让你能够无缝地将不同服务串联起来。下面列举了一些用例，它们过去实现起来颇为麻烦，而现在由于模型可以直接与远程 MCP 服务器通信，变得简单多了。

| **领域**                 | **MCP 工具解锁的用例**                                                                                                                                                  | **以往的障碍**                                                                                                      |
|---------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------|
| **电商 / 支付**   | - 在一轮对话中将商品加入 Shopify 购物车并返回结账 URL —— `"Add the Allbirds Men’s Tree Dasher 2 in size 10"` → 购物车链接  <br> - 生成 Stripe 付款链接 | 使用函数调用意味着你必须编写自定义的 `cart_add` 或 `create_payment_link` 封装，并自行托管中继服务器。 |
| **DevOps 与代码质量**| - 向 Sentry 查询某个文件的最新错误，然后在同一对话中创建一个附带修复建议的 GitHub issue                                                    | 在一个辅助循环中串联两个不同的第三方 API，需要 webhook 胶水代码和繁琐的状态管理。                |
| **消息 / 通知** | - 通过网页搜索获取当天上午的热门足球新闻，并在一次调用中让 Twilio 将摘要以短信形式发送到某个电话号码                                                 | 需要在后端拼接两次工具调用，并自行批量组装最终的短信内容。                             |


## 工具的工作原理

从宏观上看，MCP 工具的工作方式如下：

1. 声明服务器：当你向 `tools` 数组添加一个 MCP 块时，Responses API 运行时会先检测服务器使用的是哪种传输协议——较新的“streamable HTTP”还是较旧的 HTTP-over-SSE 变体——并使用该协议进行通信。
2. 导入工具列表：运行时会调用服务器的 `tools/list`，并传入你提供的所有请求头（API 密钥、OAuth 令牌等）。然后它将结果写入模型上下文中的一个 `mcp_list_tools` 条目。只要该条目存在，就不会再次获取列表。你可以使用 `allowed_tools` 限制模型能看到的内容。
    
    每次请求结束后，OpenAI 都会丢弃请求头的值，并且对 MCP `server_url` 只保留 scheme、域名和子域名，其余部分全部丢弃。授权密钥和服务器 URL 必须随每次 API 调用一起提供。这些值不会出现在响应对象中。schema 会尽可能使用“strict”模式，否则按原样加载。
    
3. 调用并批准工具：一旦模型知道了可用的操作，就可以调用其中之一。每次调用都会生成一个 `mcp_tool_call` 条目，默认情况下流会暂停，等待你的明确批准；不过一旦你信任该服务器，就可以关闭这个设置。
    
    批准后，运行时执行调用并以流式方式返回结果，然后由模型决定是继续串联调用另一个工具，还是返回最终答案。

## 使用 MCP 构建时的最佳实践

MCP 仍处于早期阶段，下面是一些在构建过程中可以改善模型性能和行为的最佳实践。

### 过滤工具，避免负载膨胀

远程服务器往往会暴露大量工具，却没有考虑模型将如何理解和使用它们。默认情况下，这可能导致包含数十个端点，每个端点都附带名称、描述和 JSON schema 等冗长定义，给模型上下文增加数百个 token，并增加延迟。更糟的是，许多服务器会返回完整的数据对象（例如完整的 Stripe 发票记录），即使只有少数几个字段与模型的任务相关。为了在生产环境中优化性能，请使用 Responses API 中的 `allowed_tools` 参数，限制从服务器的 `mcp_list_tools` 中纳入哪些工具。这可以减少 token 开销、缩短响应时间，并缩小模型的决策空间。你可能还想完全排除某些工具，例如能够执行写操作的工具，或涉及财务或安全影响的工具。

In [ ]:
curl https://api.openai.com/v1/responses -i \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -d '{
    "model": "gpt-4.1",
    "tools": [
        {
            "type": "mcp",
            "server_label": "gitmcp",
            "server_url": "https://gitmcp.io/openai/tiktoken",
            "allowed_tools": ["search_tiktoken_documentation", "fetch_tiktoken_documentation"],
            "require_approval": "never"
        }
    ],
    "input": "how does tiktoken work?"
}'

### 通过缓存降低延迟和 token 消耗，并将推理模型留给高复杂度任务

模型第一次连接服务器时，会为你添加的每个 MCP 服务器创建一个类型为 `mcp_list_tools` 的新条目。只要该条目存在于模型上下文中，我们就不会再次调用服务器的 `tools/list`。这类似于在用户对话级别进行缓存。如果 `mcp_list_tools` 不存在，我们会再次从 MCP 服务器导入工具列表。在后续 API 请求中传入 `previous_response_id`，是确保后续轮次中 `mcp_list_tools` 条目存在于模型上下文中的一种方式。你也可以手动将这些条目传入新的响应。另一个会影响延迟和输出 token 数量的因素是是否使用推理模型，因为推理模型会生成多得多的输出 token 以及推理 token。以下面两个示例 curl 为例，它们比较了使用和不使用推理模型时生成的 token 数量：

场景 1：非推理模型

In [ ]:
curl https://api.openai.com/v1/responses \   
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -d '{
    "model": "gpt-4.1",
    "tools": [
      {
        "type": "mcp",
        "server_label": "gitmcp",
        "server_url": "https://gitmcp.io/openai/tiktoken",
        "require_approval": "never"
      }
    ],
    "input": "how does tiktoken work?" 
  }'

In [ ]:
  "usage": {
    "input_tokens": 280,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 665,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 945
  }

场景 2：不带 `previous_response_id` 的推理模型

In [ ]:
curl https://api.openai.com/v1/responses \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -d '{
    "model": "o4-mini",
    "tools": [
      {
        "type": "mcp",
        "server_label": "gitmcp",
        "server_url": "https://gitmcp.io/openai/tiktoken",
        "require_approval": "never"
      }
    ],
    "input": "how does tiktoken work?",
    "reasoning": {
      "effort": "medium",
      "summary": "auto"
    } 
  }'

In [ ]:
  "usage": {
    "input_tokens": 36436,
    "input_tokens_details": {
      "cached_tokens": 22964
    },
    "output_tokens": 1586,
    "output_tokens_details": {
      "reasoning_tokens": 576
    },
    "total_tokens": 38022 
    }

### 将 MCP 与其他工具结合使用

MCP 工具只是 tools 数组中的又一个条目，因此模型可以将它与 `code_interpreter`、`web_search_preview,` 或 `image_generation` 等其他托管工具，以及你定义的任何自定义工具无缝结合使用。你也可以同时使用多个远程 MCP 服务器。

在本示例中，我们将创建一个智能体，担任一家虚构瑜伽服装店的定价分析师：它首先从 Alo Yoga MCP 服务器拉取竞争对手当前女士短裤、瑜伽裤和背心的价格，然后通过托管的网页搜索工具获取 Uniqlo 同样三个品类的价格。它使用 Code Interpreter 分析一个已通过 Files 端点预先上传的 CSV 中上周的销售数据，计算每件商品的收入和平均订单价值。然后，它衡量每件商品的价格与刚获取的 Uniqlo 和 Alo Yoga 基准之间的差距。任何价格高于或低于市场 15% 及以上的商品都会被标记出来，最后智能体会提交一份简明的文本报告，总结这些差异和关键收入数据。

In [ ]:
system_prompt= """You are a pricing analyst for my clothing company. Please use the MCP tool 
to fetch prices from the Alo Yoga MCP server for the categories of women's 
shorts, yoga pants, and  tank tops. Use only the MCP server for Alo yoga data, don't search the web. 

Next, use the web search tool to search for Uniqlo prices for women's shorts, yoga pants, and tank tops. 

In each case for Alo Yoga and Uniqlo, extract the
price for the top result in each category. Also provide the full URLs
 
Using the uploaded CSV file of sales data from my store, and with the 
code interpreter tool calculate revenue by product item, compute average order-value on a transaction level, and calculate the percentage price gap between the CSV data and Uniqlo/Alo Yoga prices. 
Flag products priced 15% or more above or below the market. 
Create and output a short report including the findings.

# Steps

1. **Fetch Alo Yoga Prices:**
   - Use the Alo Yoga MCP server to fetch prices for the following products:
High-Waist Airlift Legging
Sway Bra Tank
 5" Airlift Energy Short

- Ensure you find prices for each. 
- Extract the price of the top result for each category.
- include  URL links 


2. **Query Uniqlo Prices:**
   - Use the Web-Search tool to search non-sale prices for the following Uniqlo products: 
Women's AIRism Soft Biker Shorts
Women's AIRism Soft Leggings
Women's AIRism Bra Sleeveless Top
- Ensure you find non-sale prices for each. 
- Extract the price for the top result in each category.
- include  URL links 

3. **Sales Data Analysis:**
   - Use the uploaded CSV sales data to calculate revenue across each 
   product item.
   - Determine the average order-value on a transaction level.
   - For each SKU, compute the percentage price gap between the 
   CSV data and Uniqlo/Alo Yoga prices.
   - Flag products priced ≥ 15% above or below the market.

4. **Report:**
   - Compile and output a report including the flagging results

# Output Format
- A short text report explaining:
  - Any products that are priced ≥ 15% above or below the market, 
  with specific details. """

下面是一个示例 curl，其中用占位符代替了上面的系统提示词。

In [ ]:
curl https://api.openai.com/v1/responses \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -d '{
    "model": "gpt-4.1",
    "input": [
      {
        "role": "system",
        "content": [
          {
            "type": "input_text",
            "text": "ABOVE_SYSTEM_PROMPT"
          }
        ]
      }
    ],
    "tools": [
      {
        "type": "web_search_preview",
        "user_location": {
          "type": "approximate",
          "country": "US"
        },
        "search_context_size": "medium"
      },
      {
        "type": "code_interpreter",
        "container": {
          "type": "auto",
          "file_ids": [
            "file-WTiyGcZySaU6n218gj4XxR"
          ]
        }
      },
      {
        "type": "mcp",
        "server_url": "https://www.aloyoga.com/api/mcp",
        "server_label": "aloyoga",
        "allowed_tools": [
          "search_shop_catalog",
          "get_product_details"
        ],
        "require_approval": "never"
      }
    ],
    "temperature": 1,
    "max_output_tokens": 2048,
    "top_p": 1,
    "store": true
  }'


模型能够将 MCP 工具和网页搜索的结果延续到 Code Interpreter 步骤中，生成一份包含以下内容、并为便于阅读而进行了格式化的报告：

---
#### **价格比较与收入分析报告**

**本店销售与价格分析**

- **按商品统计的收入：**
    - 短裤：$6,060
    - 背心：$6,150
    - 瑜伽裤：$12,210
- **平均订单价值：** $872.14
- **本店各品类平均售价：**
    - 短裤：$60.00
    - 背心：$75.00
    - 瑜伽裤：$110.00

#### **与市场的价格差距**

| 品类 | 本店平均价格 | 与 Alo Yoga 的差距 (%) | 标记 (≥15%) | 与 Uniqlo 的差距 (%) | 标记 (≥15%) |
| --- | --- | --- | --- | --- | --- |
| 短裤 | $60.00 | -31.8% | **是** | +100.7% | **是** |
| 背心 | $75.00 | -14.8% |  | +114.9% | **是** |
| 瑜伽裤 | $110.00 | -14.1% |  | +267.9% | **是** |

#### **建议与标记**

**被标记的商品（价格差距 ≥15%）：**

- **短裤：** 价格比 Alo Yoga 低 31.8%，但比 Uniqlo 高 100.7%。
- **背心：** 价格比 Uniqlo 高出 114.9% 以上。
- **瑜伽裤：** 价格比 Uniqlo 高 267.9%。

短裤的定价明显低于高端竞争对手（Alo Yoga），但远高于平价替代品（Uniqlo）。如果你想在高端市场竞争，可以考虑提价；如果你想瞄准注重价格的买家，降价也是合理的。你的大多数背心和瑜伽裤也处于类似的定位——远低于 Alo，但明显高于 Uniqlo。

___

### 改进 MCP 工具调用的提示词指南

根据你的用例，你可能会发现模型会发起大量 MCP 调用，例如在使用目录搜索工具时。为了避免在庞大的商品库存上无休止地迭代，可以指示模型将搜索限制在 N 个条目以内，并且只有在用户明确要求更多信息时才提出继续搜索。这样可以让回复保持聚焦、响应迅速。

如果你使用的 MCP 服务器包含详尽的 `mcp_list_tools`，那么也值得加入一些有针对性的少样本示例，向模型展示如何选择正确的服务器，并在获得所需信息后停止，而不是发出冗余调用。

最后，添加一些指导，提醒模型：如果用户查询中缺少关键信息（尺码、颜色、产品线等），应该提出追问，而不是发起大范围搜索。这个小小的提示词调整可以减少不必要的工具调用并提升回答质量。下面是一个示例提示词，展示了如何将这些指南结合在一起：

In [ ]:
system_prompt= """You are an AI assistant that can call the following MCP servers:

1. allbirds_store
2. gitmcp

Steps
1. Use the MCP servers above to answer the user query, not every MCP server will relevant for a given query so you can choose which ones to invoke.  
2. If the user’s request lacks essential details (size, color, etc.), ask a follow-up question first rather than guessing and using the MCP server tool prematurely. 
3. If searching for products in a catalog, only return 4 results at a time before asking the user whether they want to see more. 

### Few-shot examples

# EX1 — Direct product search, then fetch variant details
user: Do you have the Allbirds Tree Runner in men’s size 10?  
assistant: {"name":"allbirds_store.search_shop_catalog",
            "arguments":{"query":"Tree Runner",
                         "context":"Men’s size 10 shoes"}}
# (assume the search returns product_id "gid://shopify/Product/987")  
assistant: {"name":"allbirds_store.get_product_details",
            "arguments":{"product_id":"gid://shopify/Product/987",
                         "options":{"Size":"10"}}}

# EX2 — Clarify missing size, then two-step color variant lookup
user: I want the Tree Runner in blue.  
assistant: Sure — what size do you need?  
user: Size 10.  
assistant: {"name":"allbirds_store.search_shop_catalog",
            "arguments":{"query":"Tree Runner",
                         "context":"Blue variant, size 10"}}
# (assume the search returns product_id "gid://shopify/Product/987")  
assistant: {"name":"allbirds_store.get_product_details",
            "arguments":{"product_id":"gid://shopify/Product/987",
                         "options":{"Size":"10","Color":"Blue"}}}

# EX3 — Git docs: search then fetch specific file
user: Can you show me how tiktoken does byte-pair encoding?  
assistant: {"name":"gitmcp.search_website_documentation",
            "arguments":{"query":"tiktoken byte-pair encoding"}}
# (assume the search returns document_id "docs/ENCODING.md")  
assistant: {"name":"gitmcp.fetch_website_documentation",
            "arguments":{"document_id":"docs/ENCODING.md"}} """


In [ ]:
curl https://api.openai.com/v1/responses \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -d '{
    "model": "gpt-4.1",
    "input": [
      {
        "role": "system",
        "content": [
          {
            "type": "input_text",
            "text": "ABOVE_SYSTEM_PROMPT"
          }
        ]
      },
      {
        "role": "user",
        "content": [
          {
            "type": "input_text",
            "text": "find me womens tree loungers in size 8"
          }
        ]
      }
    ],
    "tools": [
      {
        "type": "mcp",
        "server_url": "https://www.allbirds.com/api/mcp",
        "server_label": "allbirds",
        "allowed_tools": [
          "search_shop_catalog",
          "get_cart",
          "update_cart",
          "search_shop_policies_and_faqs",
          "get_product_details"
        ],
        "require_approval": "never"
      },
      {
        "type": "mcp",
        "server_label": "gitmcp",
        "server_url": "https://gitmcp.io/openai/tiktoken",
        "allowed_tools": [
          "fetch_tiktoken_documentation",
          "search_tiktoken_documentation",
          "search_tiktoken_code",
          "fetch_generic_url_content"
        ],
        "require_approval": "never"
      }
    ],
    "temperature": 1,
    "max_output_tokens": 2048
  }'


## 结论

托管的 MCP 工具让外部服务可以通过 Responses API 使用。连接一个远程服务器并让运行时缓存其工具列表，就无需在应用代码中管理那一次网络跳转。使用 `allowed_tools` 可以减少 token 开销，并为模型提供一个简洁、易于发现的操作集合。你可以将 MCP 与 `code_interpreter`、`web_search_preview` 或 `image_gen` 等内置工具结合，构建多服务工作流。